# 01 CDRMIP tasmax Pipeline

In [1]:
%load_ext autoreload
%autoreload 2
    
from dask.distributed import Client
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

from cdrmip_extremes import configs, pipeline, io_disk, file_search, processing, load_raw
from cdrmip_extremes.extremes_pipeline import ExtremesPipeline
from cdrmip_extremes.configs import PROC_DIR

In [2]:
client = Client(memory_limit=None,threads_per_worker=1,n_workers=28)

In [3]:
config = pipeline.PipelineConfig(
    experiments=[
        pipeline.ExperimentConfig(
            experiment_id='1pctCO2-cdr', table_id='day', variable_id='tasmax',
            anomaly_reference='picontrol',grid_label=None,
            crossing_gwls=[1.5], overshoot=True,
            gwl_match_period=(320, 340),
            rolling_window=21,
            reference_period=(None,None)
        ),
    ],
    realign_years_overrides={'1pctCO2': (0, 140)}
)

In [4]:
config = pipeline.CDRMIPPipeline(config)

In [5]:
file_search.find_CMIP_files(
    experiment_id='1pctCO2-cdr',
    table_id='day',
    variable_id='tasmax'
)

,activity_id,institution_id,source_id,experiment_id,member_id,table_id,variable_id,grid_label,version,path
0,CDRMIP,CSIRO,ACCESS-ESM1-5,1pctCO2-cdr,r1i1p1f1,day,tasmax,gn,latest,[/g/data/fs38/publications/CMIP6/CDRMIP/CSIRO/...
1,CDRMIP,NCC,NorESM2-LM,1pctCO2-cdr,r1i1p1f1,day,tasmax,gn,v20210118,[/g/data/oi10/replicas/CMIP6/CDRMIP/NCC/NorESM...
2,CDRMIP,MOHC,UKESM1-0-LL,1pctCO2-cdr,r1i1p1f2,day,tasmax,gn,v20200425,[/g/data/oi10/replicas/CMIP6/CDRMIP/MOHC/UKESM...
3,CDRMIP,MIROC,MIROC-ES2L,1pctCO2-cdr,r1i1p1f2,day,tasmax,gn,v20220506,[/g/data/oi10/replicas/CMIP6/CDRMIP/MIROC/MIRO...
4,CDRMIP,CCCma,CanESM5,1pctCO2-cdr,r1i1p2f1,day,tasmax,gn,v20190429,[/g/data/oi10/replicas/CMIP6/CDRMIP/CCCma/CanE...
5,CDRMIP,CNRM-CERFACS,CNRM-ESM2-1,1pctCO2-cdr,r1i1p1f2,day,tasmax,gr,v20210121,[/g/data/oi10/replicas/CMIP6/CDRMIP/CNRM-CERFA...


In [ ]:
pipe = config.load_raw()

=== Loading 1pctCO2-cdr ===
Searching for files matching: {'experiment_id': '1pctCO2-cdr', 'table_id': 'day', 'variable_id': 'tasmax', 'grid_label': None, 'member_id': None}
Loading ACCESS-ESM1-5 / r1i1p1f1
  0241-01-01 12:00:00 -> 1000-12-31 12:00:00
Loading CNRM-ESM2-1 / r1i1p1f2
  1990-01-01 12:00:00 -> 2189-12-31 12:00:00
Loading CanESM5 / r1i1p2f1
  1991-01-01 12:00:00 -> 2290-12-31 12:00:00
Loading MIROC-ES2L / r1i1p1f2


In [ ]:
pipe.concatenate_to_parents()

In [ ]:
pipe.load_picontrol()

In [ ]:
pipe.regrid()

In [ ]:
pipe.compute_txx()

In [ ]:
pipe.compute_txx_anomaly()
pipe.txx_anomaly

In [ ]:
pipe.compute_tx90p()

## Save

In [17]:
io_disk.save_tree(
    pipe.txx_anomaly,
    PROC_DIR,
    'txx',
    activity_id='CDRMIP',
    category='anomaly',
    overwrite=True
)

Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/txx/ACCESS-ESM1-5/1pctCO2-cdr/ACCESS-ESM1-5_1pctCO2-cdr_r1i1p1f1_txx_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/txx/CNRM-ESM2-1/1pctCO2-cdr/CNRM-ESM2-1_1pctCO2-cdr_r1i1p1f2_txx_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/txx/CanESM5/1pctCO2-cdr/CanESM5_1pctCO2-cdr_r1i1p2f1_txx_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/txx/MIROC-ES2L/1pctCO2-cdr/MIROC-ES2L_1pctCO2-cdr_r1i1p1f2_txx_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/txx/UKESM1-0-LL/1pctCO2-cdr/UKESM1-0-LL_1pctCO2-cdr_r1i1p1f2_txx_anomaly.nc


In [ ]:
io_disk.save_tree(
    pipe.tx90p,
    PROC_DIR,
    'tx90p',
    activity_id='CDRMIP',
    category=None
)

In [19]:
from intake_esgf import ESGFCatalog
cat = ESGFCatalog()
cat.search(
    experiment_id="1pctCO2",
    source_id="NorESM2-LM",
    table_id="day",
    variable_id="tasmax",
)
dsd = cat.to_dataset_dict()

In [21]:
  # dict of xr.Datasets, keyed by dataset id

INFO:intake-esgf:search begin experiment_id=['1pctCO2'], source_id=['NorESM2-LM'], table_id=['day'], variable_id=['tasmax'], type=['Dataset'], project=['CMIP6'], latest=[True], retracted=[False]


The history saving thread hit an unexpected error (OperationalError('disk I/O error')).History will not be written to the database.


   Searching indices:   0%|          |0/1 [       ?index/s]

INFO:intake-esgf:└─GlobusESGFIndex('ESGF2-US-1.5-Catalog') results=1 response_time=0.02
INFO:intake-esgf:combine_time=0.01
INFO:intake-esgf:search end total_time=0.04
INFO:intake-esgf:file info begin
INFO:intake-esgf:└─GlobusESGFIndex('ESGF2-US-1.5-Catalog') results=15 response_time=0.01


Get file information:   0%|          |0/1 [       ?index/s]

...

tasmax_day_NorESM2-LM_1pctCO2_r1i1p1f...:   0%|          |0.00/120M [?B/s]

tasmax_day_NorESM2-LM_1pctCO2_r1i1p1f...:   0%|          |0.00/120M [?B/s]

tasmax_day_NorESM2-LM_1pctCO2_r1i1p1f...:   0%|          |0.00/120M [?B/s]

INFO:intake-esgf:download failed https://g-52ba3.fd635.8443.data.globus.org/css03_data/CMIP6/CMIP/NCC/NorESM2-LM/1pctCO2/r1i1p1f1/day/tasmax/gn/v20190815/tasmax_day_NorESM2-LM_1pctCO2_r1i1p1f1_gn_01410101-01501231.nc
INFO:intake-esgf:download failed https://g-52ba3.fd635.8443.data.globus.org/css03_data/CMIP6/CMIP/NCC/NorESM2-LM/1pctCO2/r1i1p1f1/day/tasmax/gn/v20190815/tasmax_day_NorESM2-LM_1pctCO2_r1i1p1f1_gn_01310101-01401231.nc
INFO:intake-esgf:download failed https://g-52ba3.fd635.8443.data.globus.org/css03_data/CMIP6/CMIP/NCC/NorESM2-LM/1pctCO2/r1i1p1f1/day/tasmax/gn/v20190815/tasmax_day_NorESM2-LM_1pctCO2_r1i1p1f1_gn_01210101-01301231.nc


MissingFileInformation: We were unable to find file information for these keys: {'CMIP6.CMIP.NCC.NorESM2-LM.1pctCO2.r1i1p1f1.day.tasmax.gn'}. Your access options could affect the possibilties.